# 10 · Prévision des ventes

**Projet 4 : Analyse des données de vente et retours** · Sprint 5 · Unité d'apprentissage 3

**Question métier** : *combien chaque boutique vendra-t-elle le mois prochain et les mois suivants, et qu'est-ce qui fait monter ou baisser cette prévision ?*

**Objectif de l'énoncé servi par ce notebook** : prédire les ventes futures, pour préparer les recommandations du volet prescriptif. Le modèle retenu sert ensuite de moteur au simulateur du notebook 14.

**Le cadre**

| Choix | Décision | Pourquoi |
|---|---|---|
| Cible | Chiffre d'affaires net du mois, en logarithme | L'écart de taille entre la boutique en ligne (jusqu'à 27 M$ par mois) et un petit magasin est énorme ; les erreurs sont reconverties en dollars pour la lecture |
| Granularité | Une ligne par boutique et par mois | C'est le niveau où se prennent les décisions et où le résultat est déjà calculé (`05b_charges`) |
| Entraînement | Février 2019 à décembre 2020 | Janvier 2019 n'a pas de mois précédent |
| Test | 2021, l'exercice étudié | Jamais vu par le modèle avant la partie 7 |
| Règle anti-fuite | Chaque variable d'un mois est connue au début de ce mois | Les ventes du mois ne servent jamais à le prévoir, seulement celles des mois précédents |

| Étape | Ce qu'on fait | Question posée |
|---|---|---|
| 1-2. Table de modélisation | Variables candidates en six familles, corrélations | Que sait-on au début d'un mois ? |
| 3. Validation | Plis glissants par mois, à un mois et à 6 mois | Comment juger un modèle sans regarder 2021 ? |
| 4. Cible | Niveau ou variation du chiffre d'affaires | Que doit apprendre le modèle ? |
| 5. Sélection des variables | Retrait par famille, permutation, élimination pas à pas | Quelles variables apportent assez pour leur coût ? |
| 6. Modèles | Ridge linéaire et quadratique (grille), Random Forest, gradient boosting, XGBoost (Optuna) | Quel modèle prévoit le mieux, aux deux horizons ? |
| 7. Test sur 2021 | À un mois et à douze mois, avec et sans assortiment | Le modèle bat-il les références naïves ? |
| 8. Interprétation | Permutation, SHAP, forme de la relation, erreurs | Que regarde le modèle, et où se trompe-t-il ? |
| 9. Prévision 2022 | Modèle réentraîné sur tout l'historique, hypothèses au fil de l'eau | Que vendrait l'entreprise si rien ne change ? |
| 10. Enregistrement | Prévisions, modèle, MLflow, tableaux | Que réutilisent le simulateur et l'application ? |

Les scénarios chiffrés (« que se passe-t-il si l'on lance plus de produits ? ») ne sont **pas** traités ici : ils relèvent des notebooks prescriptifs 13 et 14.

## 0. Préparation

In [ ]:
import os
import sys
import warnings
from pathlib import Path

os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")      # messages d'aide de MLflow inutiles ici
os.environ.setdefault("GIT_PYTHON_REFRESH", "quiet")
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

import joblib
import numpy as np
import pandas as pd
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import optuna
import mlflow
import mlflow.sklearn
import shap
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures, StandardScaler
from xgboost import XGBRegressor

# ---------------------------------------------------------------------------
# 1. Retrouver la racine du projet (le dossier qui contient src/ et data/)
# ---------------------------------------------------------------------------
racine = Path.cwd()
while not ((racine / "src").is_dir() and (racine / "data").is_dir()):
    if racine == racine.parent:
        raise FileNotFoundError("Racine du projet introuvable : lancer ce notebook depuis le dossier Projet ADD ou un de ses sous-dossiers.")
    racine = racine.parent
sys.path.insert(0, str(racine))

# ---------------------------------------------------------------------------
# 2. Modules du projet
# ---------------------------------------------------------------------------
#   config  -> chemins, années de l'étude
#   viz     -> charte graphique
#   analyse -> libellés français, export des tableaux
#   modeles -> table de modélisation, mesures d'erreur, prévision récursive (réutilisés par le simulateur 14)
from src import config, viz, analyse, modeles
from src.analyse import LIBELLES_TYPES, ORDRE_TYPES, NOMS_MOIS

config.creer_dossiers()
viz.appliquer_style()
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ---------------------------------------------------------------------------
# 3. Exercices et paramètres
# ---------------------------------------------------------------------------
ANNEES = config.ANNEES_ETUDE                  # [2019, 2020, 2021]
EXERCICE = config.EXERCICE                    # 2021 : l'exercice de test, jamais vu à l'entraînement
GRAINE = 2026                                 # graine fixe : mêmes résultats à chaque exécution
ESSAIS_OPTUNA = 25                            # essais Optuna par modèle à arbres
COULEURS_TYPES = {"En ligne": viz.CATEGORIES[2], "Revendeur": viz.ACCENT, "Magasins": viz.ALERTE}

# ---------------------------------------------------------------------------
# 4. MLflow : un carnet de bord des entraînements, dans mlruns/ (exclu de Git)
# ---------------------------------------------------------------------------
#    Pour consulter les essais, depuis le dossier du projet :
#    mlflow ui --backend-store-uri sqlite:///mlruns/mlflow.db       puis ouvrir http://127.0.0.1:5000
DOSSIER_MLFLOW = config.ROOT / "mlruns"
DOSSIER_MLFLOW.mkdir(exist_ok=True)
mlflow.set_tracking_uri("sqlite:///" + (DOSSIER_MLFLOW / "mlflow.db").as_posix())
EXPERIENCE = "10_prevision_ventes"
if mlflow.get_experiment_by_name(EXPERIENCE) is None:
    mlflow.create_experiment(EXPERIENCE, artifact_location=(DOSSIER_MLFLOW / "artefacts").as_uri())
mlflow.set_experiment(EXPERIENCE)


def m_dollars(valeur, decimales=1, signe=False):
    """Montant en millions de dollars, virgule française : 12,3 M$."""
    texte = f"{valeur / 1e6:{'+' if signe else ''},.{decimales}f}".replace(",", " ").replace(".", ",")
    return f"{texte} M$"


def pct(valeur, decimales=1, signe=False):
    """Taux en pourcentage, virgule française : 0.07 -> 7,0 %."""
    return f"{valeur:{'+' if signe else ''}.{decimales}%}".replace(".", ",").replace("%", " %")


tableaux = {}                                 # tableaux exportés vers Excel à la fin
print("Racine du projet :", config.ROOT)
print("Versions :", "scikit-learn", sklearn.__version__, "· optuna", optuna.__version__, "· mlflow", mlflow.__version__, "· shap", shap.__version__)

## 1. Chargement

In [ ]:
chemins = {
    "commun": config.AGREGATS / "ventes_charges_boutique_mois.parquet",
    "ventes": config.CLEAN / "fait_ventes_variables.parquet",
    "dim_boutique": config.CLEAN_DIMENSIONS / "dim_boutique.parquet",
    "dim_produit": config.CLEAN_DIMENSIONS / "dim_produit.parquet",
    "contexte": config.CLEAN_DIMENSIONS / "contexte_pays_date.parquet",
}
for nom, chemin in chemins.items():
    if not chemin.exists():
        raise FileNotFoundError(f"{chemin} introuvable : exécuter d'abord les notebooks 00 à 05c.")

commun = pd.read_parquet(chemins["commun"])
ventes = pd.read_parquet(chemins["ventes"], columns=["id_boutique", "date", "taux_remise_annonce"])
dim_boutique = pd.read_parquet(chemins["dim_boutique"])
dim_produit = pd.read_parquet(chemins["dim_produit"])
contexte = pd.read_parquet(chemins["contexte"])

print(f"Table ventes et charges : {len(commun):,} lignes boutique × mois, {commun.id_boutique.nunique()} boutiques, "
      f"{commun.date.min():%Y-%m} à {commun.date.max():%Y-%m}".replace(",", " "))

## 2. La table de modélisation

`src/modeles.py` assemble une ligne par boutique et par mois. Les mêmes fonctions serviront au simulateur du notebook 14, pour que le modèle reçoive toujours des variables calculées de la même façon.

| Famille | Variables | Connue au début du mois parce que… |
|---|---|---|
| Calendrier | Saison (mois placé sur un cercle : `mois_sin`, `mois_cos`), nombre de jours, de jours de fin de semaine, de jours fériés du pays | C'est le calendrier |
| Contexte COVID | Sévérité moyenne des mesures du pays (OxCGRT) | Variable externe, publiée au jour le jour |
| Boutique | Type, surface, effectif | Caractéristiques fixes |
| Historique | Log du CA du mois précédent, moyenne des 3 mois précédents | Ventes passées seulement |
| Assortiment | Produits lancés sur les 3 et les 12 derniers mois, âge moyen du catalogue | La date de mise en vente fait partie du plan de lancement |
| Commercial | Taux annoncé de la promotion du mois, budget publicitaire de la boutique | Calendrier promotionnel et budget fixés à l'avance |

Le « même mois l'an dernier » n'est **pas** une variable du modèle : il n'existe pas pour 2019, et l'utiliser aurait réduit l'entraînement à la seule année 2020. Il sert seulement de référence naïve.

In [ ]:
table = modeles.construire_table(commun, dim_boutique, dim_produit, contexte, ventes)
table["type"] = table.type_boutique.map(LIBELLES_TYPES)

CANDIDATES = [v for famille in modeles.FAMILLES.values() for v in famille]
NUMERIQUES_CANDIDATES = [v for v in CANDIDATES if v not in modeles.VARIABLES_CATEGORIELLES]

# Le premier mois de chaque boutique n'a pas de mois précédent : il ne peut pas servir
sans_historique = table.log_ca_m1.isna()
print(f"Lignes boutique × mois : {len(table):,}, dont {sans_historique.sum()} premiers mois sans mois précédent, retirés".replace(",", " "))
table = table[~sans_historique].reset_index(drop=True)

manquants = table[CANDIDATES].isna().sum()
print("Valeurs manquantes dans les variables candidates :", int(manquants.sum()))

familles = pd.DataFrame([(f, v) for f, vs in modeles.FAMILLES.items() for v in vs], columns=["Famille", "Variable"])
display(familles.groupby("Famille", sort=False).Variable.agg(", ".join).to_frame("Variables candidates"))

resume = table.groupby("annee").agg(lignes=("ca_net", "size"), boutiques=("id_boutique", "nunique"),
                                    ca_net=("ca_net", "sum"), lancements_12m_moyen=("lancements_12m", "mean"),
                                    age_catalogue_moyen=("age_catalogue", "mean"), severite_covid=("severite_covid", "mean"))
resume["ca_net"] = resume.ca_net.map(m_dollars)
display(resume)

**Lecture.** La table compte 1 919 lignes utilisables : les 60 premiers mois de boutique n'ont pas de mois précédent. Les variables d'assortiment résument déjà la thèse de l'UA2 : 588 produits lancés sur douze mois en moyenne en 2019, 478 en 2020, 351 en 2021, et un catalogue qui vieillit de 16 à 20 mois d'âge moyen. Ces chiffres sont les mêmes pour toutes les boutiques un mois donné : le modèle ne voit donc que 23 valeurs différentes de ces variables à l'entraînement, une par mois. C'est la limite principale de tout ce notebook.

### Corrélations entre variables candidates

Deux variables très corrélées disent la même chose : le modèle peut attribuer à l'une ce qui revient à l'autre, et l'interprétation devient fragile.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7.5))
correlations = table[NUMERIQUES_CANDIDATES + ["variation"]].corr()
sns.heatmap(correlations, cmap=sns.diverging_palette(15, 160, s=60, as_cmap=True), center=0, vmin=-1, vmax=1,
            annot=True, fmt=".2f", annot_kws={"size": 7}, cbar_kws={"shrink": 0.7}, ax=ax)
viz.titre(ax, "Corrélations entre variables candidates", "Boutique × mois, février 2019 à décembre 2021 · variation = log du CA du mois moins celui du mois précédent")
viz.enregistrer(fig, "10_correlations_variables")
plt.show()

paires = (correlations.where(np.triu(np.ones(correlations.shape, dtype=bool), k=1)).stack()
          .rename("corr").reset_index().assign(abs=lambda d: d["corr"].abs()).sort_values("abs", ascending=False))
print("Paires de variables les plus corrélées :")
display(paires.head(8).drop(columns="abs").rename(columns={"level_0": "Variable 1", "level_1": "Variable 2"}))

**Lecture.**
- **Historique** : le mois précédent et la moyenne des 3 mois précédents sont presque identiques (0,97). L'un des deux suffira.
- **Promotion et saison** : le taux de promotion suit la saison (0,89 avec `mois_cos`), puisque le calendrier promotionnel est le même chaque année (notebook 07).
- **Assortiment et temps** : les lancements sur 12 mois sont corrélés à −0,72 avec la sévérité COVID et à −0,83 avec l'âge du catalogue. Les trois variables décrivent la même chose, le passage de 2019 à 2021. Le modèle ne pourra pas séparer proprement « moins de nouveautés » et « période COVID » : ce point revient dans l'interprétation (partie 8).
- **Contexte COVID** : Contoso couvre en réalité 2007 à 2009 et n'a jamais connu les confinements (rapport L3 + L4). Ici, l'indice joue le rôle d'un **marqueur de période** (avant ou après 2020), pas d'un effet sanitaire.
- Aucune variable n'est fortement corrélée à la variation du chiffre d'affaires d'un mois sur l'autre (au plus −0,35 avec le nombre de jours) : le mois suivant se prévoit mal, ce que la suite confirme.

## 3. Découpage dans le temps et validation

**Le test** : 2021 est mis de côté et n'est utilisé qu'en partie 7, une seule fois.

**La validation** se fait uniquement sur 2019 et 2020, par plis glissants : chaque pli apprend sur tous les mois passés et se note sur les mois suivants, jamais l'inverse. Le découpage porte sur les mois et non sur les lignes, pour que les boutiques d'un même mois tombent du même côté.

Le modèle aura deux usages, donc deux critères :
- **à un mois** : prévoir le mois suivant en connaissant le mois qui vient de se terminer (suivi mensuel) ;
- **à moyen terme** : prévoir plusieurs mois d'un coup, chaque prévision servant d'historique au mois suivant (prévision de l'année, simulateur).

Le **score** de validation est la moyenne des deux. Il sert à toutes les décisions : cible, variables, hyperparamètres, choix du modèle. L'erreur est le **WAPE** : la somme des écarts en dollars rapportée au chiffre d'affaires total (10 % = on se trompe de 10 % des ventes).

In [ ]:
entrainement = table[table.annee < EXERCICE].reset_index(drop=True)
test = table[table.annee == EXERCICE].reset_index(drop=True)


def plis_temporels(lignes, plis, mois_par_pli):
    """Découpage par mois : chaque pli apprend sur tous les mois passés et se note sur les mois suivants.

    Le découpage se fait sur les mois, pas sur les lignes : toutes les boutiques d'un même mois tombent du même côté.
    """
    mois = np.array(sorted(lignes.date.unique()))
    resultat = []
    for k in range(plis, 0, -1):
        fin = len(mois) - k * mois_par_pli
        resultat.append((np.where(lignes.date.isin(mois[:fin]))[0],
                         np.where(lignes.date.isin(mois[fin:fin + mois_par_pli]))[0]))
    return resultat


PLIS_COURT = plis_temporels(entrainement, 4, 3)       # prévision à un mois, notée sur 4 fenêtres de 3 mois
PLIS_LONG = plis_temporels(entrainement, 2, 6)        # prévision de 6 mois d'un coup, notée sur 2 fenêtres de 6 mois


def decrire(plis, critere):
    return pd.DataFrame([{
        "Critère": critere, "Pli": i + 1,
        "Apprentissage": f"{entrainement.date.iloc[a].min():%Y-%m} à {entrainement.date.iloc[a].max():%Y-%m}",
        "Mois appris": entrainement.date.iloc[a].nunique(),
        "Validation": f"{entrainement.date.iloc[v].min():%Y-%m} à {entrainement.date.iloc[v].max():%Y-%m}",
        "Lignes notées": len(v)} for i, (a, v) in enumerate(plis)])


print(f"Entraînement : {len(entrainement):,} lignes ({entrainement.date.min():%Y-%m} à {entrainement.date.max():%Y-%m}) · "
      f"test : {len(test):,} lignes ({EXERCICE}), jamais utilisées avant la partie 7".replace(",", " "))
display(pd.concat([decrire(PLIS_COURT, "Court terme : à un mois"), decrire(PLIS_LONG, "Moyen terme : 6 mois d'un coup")])
        .set_index(["Critère", "Pli"]))

## 4. Choix de la cible : le niveau ou la variation ?

Deux façons d'apprendre le chiffre d'affaires d'un mois :
- **le niveau** : le log du chiffre d'affaires du mois ;
- **la variation** : l'écart de ce log avec celui du mois précédent. Ce choix enlève la taille de la boutique et permet de prévoir des niveaux jamais vus, ce qu'un modèle à arbres ne sait pas faire avec le niveau.

Les deux sont comparées avec une Random Forest de référence (150 arbres) et toutes les variables candidates, sur les deux critères de validation. La référence naïve prévoit le mois précédent à un mois, et reconduit le dernier mois connu à moyen terme.

In [ ]:
def pipeline(nom, variables, **params):
    """Préparation des variables puis modèle : un seul objet, qui s'entraîne et prévoit d'un bloc."""
    numeriques = [v for v in variables if v not in modeles.VARIABLES_CATEGORIELLES]
    categorielles = [v for v in variables if v in modeles.VARIABLES_CATEGORIELLES]
    num = (make_pipeline(StandardScaler(), PolynomialFeatures(2, include_bias=False), StandardScaler())
           if nom == "Ridge quadratique" else StandardScaler())
    etapes = [("num", num, numeriques)]
    if categorielles:
        etapes.append(("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorielles))
    preparation = ColumnTransformer(etapes, verbose_feature_names_out=False)
    if nom in ("Ridge linéaire", "Ridge quadratique"):
        modele = Ridge(**params)
    elif nom == "Random Forest":
        modele = RandomForestRegressor(random_state=GRAINE, n_jobs=-1, **params)
    elif nom == "Gradient boosting (scikit-learn)":
        modele = HistGradientBoostingRegressor(random_state=GRAINE, **params)
    elif nom == "XGBoost":
        modele = XGBRegressor(random_state=GRAINE, n_jobs=-1, **params)
    else:
        raise ValueError(nom)
    return Pipeline([("preparation", preparation), ("modele", modele)])


def colonne_cible(cible):
    return "variation" if cible == "variation" else "log_ca"


def en_dollars(lignes, sortie, cible):
    """Ramène la sortie du modèle à un chiffre d'affaires en dollars."""
    return np.exp(lignes.log_ca_m1.values + sortie) if cible == "variation" else np.exp(sortie)


def prevoir_long(modele, variables, cible, appr, val, permuter=None, alea=None):
    """Prévision récursive des mois de `val` à partir de l'historique réel de `appr` ; `permuter` mélange une variable."""
    futur = val.drop(columns=["log_ca_m1", "log_ca_moy3"])
    if permuter is not None:
        futur[permuter] = alea.permutation(futur[permuter].values)
    p = modeles.predire_recursif(modele, variables, appr[["id_boutique", "date", "log_ca"]], futur, cible)
    return modeles.wape(p.ca_net, p.ca_prevu)


def valider(nom, variables, cible=None, donnees=None, **params):
    """Erreurs de validation (WAPE en dollars) : à un mois sur PLIS_COURT, à 6 mois d'un coup sur PLIS_LONG.

    Le score unique est la moyenne des deux critères : le modèle doit servir aux deux horizons
    (suivi mensuel, puis prévision de l'année et simulateur).
    """
    cible = CIBLE if cible is None else cible
    donnees = entrainement if donnees is None else donnees
    court, long_ = [], []
    for appr, val in PLIS_COURT:
        a, v = donnees.iloc[appr], donnees.iloc[val]
        m = pipeline(nom, variables, **params).fit(a[variables], a[colonne_cible(cible)])
        court.append(modeles.wape(v.ca_net, en_dollars(v, m.predict(v[variables]), cible)))
    for appr, val in PLIS_LONG:
        a, v = donnees.iloc[appr], donnees.iloc[val]
        m = pipeline(nom, variables, **params).fit(a[variables], a[colonne_cible(cible)])
        long_.append(prevoir_long(m, variables, cible, a, v))
    court, long_ = np.array(court), np.array(long_)
    return {"court": court, "long": long_, "score": (court.mean() + long_.mean()) / 2}


def valider_naif():
    """Références naïves sur les mêmes plis : mois précédent (court terme), dernier mois connu reconduit (moyen terme)."""
    court = [modeles.wape(entrainement.iloc[v].ca_net, np.exp(entrainement.iloc[v].log_ca_m1)) for _, v in PLIS_COURT]
    long_ = []
    for appr, val in PLIS_LONG:
        a, v = entrainement.iloc[appr], entrainement.iloc[val]
        dernier = a.sort_values("date").groupby("id_boutique").log_ca.last()
        v = v[v.id_boutique.isin(dernier.index)]
        long_.append(modeles.wape(v.ca_net, np.exp(v.id_boutique.map(dernier))))
    court, long_ = np.array(court), np.array(long_)
    return {"court": court, "long": long_, "score": (court.mean() + long_.mean()) / 2}


def ligne_scores(nom, r):
    return {"": nom, "À un mois (4 plis)": r["court"].mean(), "À 6 mois (2 plis)": r["long"].mean(), "Score (moyenne)": r["score"]}


PARAMS_REF = dict(n_estimators=150, min_samples_leaf=3)       # modèle de référence des choix : Random Forest
essais_cible = {"Référence naïve": valider_naif(),
                "Random Forest, cible = niveau (log du CA)": valider("Random Forest", CANDIDATES, "niveau", **PARAMS_REF),
                "Random Forest, cible = variation par rapport au mois précédent": valider("Random Forest", CANDIDATES, "variation", **PARAMS_REF)}
choix_cible = pd.DataFrame([ligne_scores(n, r) for n, r in essais_cible.items()]).set_index("")
print("WAPE de validation (plus bas = meilleur), toutes les variables candidates :")
display(choix_cible.style.format(lambda v: pct(v, 2)))
tableaux["4_choix_cible"] = choix_cible.reset_index(names="Essai")
CIBLE = "niveau" if essais_cible["Random Forest, cible = niveau (log du CA)"]["score"] <= essais_cible["Random Forest, cible = variation par rapport au mois précédent"]["score"] else "variation"
print("Cible retenue :", "log du chiffre d'affaires du mois" if CIBLE == "niveau" else "variation par rapport au mois précédent")

**Lecture.**
- **Le niveau l'emporte** : score de 20,47 %, contre 29,68 % pour la variation. La variation se trompe surtout à 6 mois (37,17 %) : en prévoyant des écarts mois après mois, le modèle additionne ses erreurs.
- **Le naïf reste dur à battre à un mois** (13,88 %, contre 15,70 % pour le niveau). Le modèle ne le dépasse qu'à 6 mois (25,25 % contre 28,17 %), là où reconduire le dernier mois ne suffit plus.
- **Le premier pli pèse lourd** : il valide sur janvier à mars 2020, la rupture où les magasins perdent 43 % de leurs ventes dès janvier (rapport L3 + L4). Aucun modèle appris sur 2019 ne pouvait la voir venir. C'est voulu : un modèle jugé seulement sur des mois calmes donnerait une image trop flatteuse.

La cible retenue est le **niveau** (log du chiffre d'affaires du mois).

## 5. Sélection des variables

Trois regards complémentaires, tous sur la validation, jamais sur 2021 :
1. **par famille** : ce que perd le modèle quand on retire une famille entière ;
2. **par permutation** : ce que perd un modèle déjà entraîné quand on mélange une variable ;
3. **par élimination pas à pas** : on réentraîne le modèle sans chaque variable, et on retire celles qui n'apportent pas assez pour leur coût. C'est ce troisième regard qui décide.

### 5.1 Par famille

In [ ]:
base = valider("Random Forest", CANDIDATES, **PARAMS_REF)
retraits = []
for famille, variables in modeles.FAMILLES.items():
    r = valider("Random Forest", [v for v in CANDIDATES if v not in variables], **PARAMS_REF)
    retraits.append({"Famille retirée": famille, "Variables": ", ".join(variables),
                     "À un mois": 100 * (r["court"].mean() - base["court"].mean()),
                     "À 6 mois": 100 * (r["long"].mean() - base["long"].mean()),
                     "Score": 100 * (r["score"] - base["score"])})
retraits = pd.DataFrame(retraits).sort_values("Score", ascending=False).reset_index(drop=True)
print(f"Avec toutes les variables : {pct(base['court'].mean(), 2)} à un mois, {pct(base['long'].mean(), 2)} à 6 mois, score {pct(base['score'], 2)}")
print("Hausse de l'erreur quand on retire la famille (en points de WAPE ; négatif = le modèle s'améliore sans elle) :")
display(retraits.style.format({c: "{:+.2f}" for c in ["À un mois", "À 6 mois", "Score"]}))
tableaux["5_retrait_familles"] = retraits

graphe = retraits.melt(id_vars="Famille retirée", value_vars=["À un mois", "À 6 mois"], var_name="Horizon", value_name="Points")
fig, ax = plt.subplots(figsize=(10, 4.2))
sns.barplot(data=graphe, y="Famille retirée", x="Points", hue="Horizon", palette=[viz.PRINCIPALE, viz.ACCENT], ax=ax)
ax.axvline(0, color=viz.GRIS, lw=1)
ax.grid(axis="x"); ax.grid(axis="y", visible=False); ax.set_ylabel(""); ax.legend(title="")
ax.set_xlabel("Hausse de l'erreur (WAPE, en points) quand on retire la famille")
viz.titre(ax, "Ce que perd le modèle sans chaque famille de variables", "Validation sur 2019 et 2020 · Random Forest de référence")
viz.enregistrer(fig, "10_retrait_familles")
plt.show()

**Lecture.**
- **Le contexte COVID est la famille la plus utile** : sans elle, l'erreur monte de 7,5 points à un mois et de 4,9 points à 6 mois. Comme vu en partie 2, elle sert de marqueur de période : elle dit au modèle qu'on est passé dans le « monde d'après janvier 2020 ».
- **La boutique** compte surtout à 6 mois (+2,4 points) : quand l'historique est lui-même prévu, sa taille et son effectif ancrent le niveau.
- **L'assortiment** n'apporte presque rien sur 2020 (+0,5 point à un mois, rien à 6 mois). Normal : sur ces plis, il varie à peine d'un mois sur l'autre et suit la même pente que l'indice COVID.
- **L'historique et le commercial** aident à un mois mais gênent à 6 mois : à moyen terme, le mois précédent est une prévision, et le modèle propage ses propres erreurs.

Ce regard par famille ne suffit pas pour décider : une famille utile peut contenir des variables inutiles, et une variable seule peut être remplacée par sa voisine.

### 5.2 Par permutation

On garde le modèle entraîné et on mélange une variable dans les données de validation : si l'erreur monte, la variable servait. C'est rapide, mais un piège est attendu ici : une variable qui ne change presque pas à l'intérieur d'un pli (l'indice COVID, les lancements) n'est presque pas modifiée par le mélange, et paraît inutile même quand elle compte.

In [ ]:
def importance_permutation(modele, lignes, variables, cible=None, repetitions=5, graine=GRAINE):
    """Hausse moyenne du WAPE à un mois quand on mélange une variable (en points) : ce que la variable apporte."""
    cible = CIBLE if cible is None else cible
    alea = np.random.default_rng(graine)
    base = modeles.wape(lignes.ca_net, en_dollars(lignes, modele.predict(lignes[variables]), cible))
    resultat = {}
    for var in variables:
        hausses = []
        for _ in range(repetitions):
            melange = lignes[variables].copy()
            melange[var] = alea.permutation(melange[var].values)
            hausses.append(modeles.wape(lignes.ca_net, en_dollars(lignes, modele.predict(melange), cible)) - base)
        resultat[var] = 100 * np.mean(hausses)
    return pd.Series(resultat)


HISTORIQUE = modeles.FAMILLES["Historique"]
alea = np.random.default_rng(GRAINE)
court, long_ = [], []
for appr, val in PLIS_COURT:
    a, v = entrainement.iloc[appr], entrainement.iloc[val]
    m = pipeline("Random Forest", CANDIDATES, **PARAMS_REF).fit(a[CANDIDATES], a[colonne_cible(CIBLE)])
    court.append(importance_permutation(m, v, CANDIDATES))
for appr, val in PLIS_LONG:
    a, v = entrainement.iloc[appr], entrainement.iloc[val]
    m = pipeline("Random Forest", CANDIDATES, **PARAMS_REF).fit(a[CANDIDATES], a[colonne_cible(CIBLE)])
    reference_long = prevoir_long(m, CANDIDATES, CIBLE, a, v)
    # À moyen terme, l'historique est produit par le modèle lui-même : on ne peut mélanger que les autres variables
    long_.append(pd.Series({var: 100 * np.mean([prevoir_long(m, CANDIDATES, CIBLE, a, v, permuter=var, alea=alea) - reference_long
                                                for _ in range(3)]) for var in CANDIDATES if var not in HISTORIQUE}))
court, long_ = pd.concat(court, axis=1), pd.concat(long_, axis=1)
importance = pd.DataFrame({"À un mois": court.mean(axis=1), "Plis où elle aide (sur 4)": (court > 0).sum(axis=1),
                           "À 6 mois": long_.mean(axis=1).reindex(court.index)})
importance["Apport"] = importance[["À un mois", "À 6 mois"]].mean(axis=1)          # l'historique : à un mois seulement
importance["Famille"] = importance.index.map({v: f for f, vs in modeles.FAMILLES.items() for v in vs})
importance = importance.sort_values("Apport", ascending=False)
display(importance.style.format({"À un mois": "{:+.2f}", "À 6 mois": "{:+.2f}", "Apport": "{:+.2f}"}, na_rep="n.d."))
tableaux["5_importance_variables"] = importance.rename_axis("Variable").reset_index()

graphe = importance.rename_axis("Variable").reset_index().melt(id_vars="Variable", value_vars=["À un mois", "À 6 mois"],
                                                              var_name="Horizon", value_name="Points").dropna()
fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=graphe, y="Variable", x="Points", hue="Horizon", order=importance.index, palette=[viz.PRINCIPALE, viz.ACCENT], ax=ax)
ax.axvline(0, color=viz.GRIS, lw=1)
ax.grid(axis="x"); ax.grid(axis="y", visible=False); ax.set_ylabel(""); ax.legend(title="")
ax.set_xlabel("Hausse de l'erreur (WAPE, en points) quand on mélange la variable")
viz.titre(ax, "Importance de chaque variable par permutation", "Moyenne des plis de validation · Random Forest de référence")
viz.enregistrer(fig, "10_importance_permutation_cv")
plt.show()

**Lecture.**
- Les variables de **niveau** dominent : le mois précédent (+45 points à un mois) et le type de boutique (+40 points), puis l'effectif, la surface et le taux de promotion.
- **Le piège se vérifie** : la sévérité COVID ressort à +0,00 point par permutation, alors que retirer la famille faisait monter l'erreur de 6,2 points. À l'intérieur de trois mois, l'indice varie peu d'une ligne à l'autre : le mélanger ne change presque rien. Les lancements sont dans le même cas.
- **Conclusion de méthode** : la permutation mesure ce qu'un modèle déjà entraîné utilise, pas ce qu'une variable apporte quand on construit le modèle. Pour sélectionner, il faut réentraîner sans la variable : c'est l'élimination pas à pas.

### 5.3 Élimination pas à pas : l'apport face au coût

À chaque tour, le modèle de référence est réentraîné sans chacune des variables restantes. **L'apport** d'une variable est la hausse du score quand on la retire.

| Règle | Seuil | Pourquoi |
|---|---|---|
| Apport minimal | 0,1 point de score | En dessous, la variable ajoute de la complexité sans gain mesurable |
| Variable coûteuse | 0,5 point | L'indice COVID se télécharge à la main et n'existe pas pour l'avenir : il doit rapporter nettement plus |
| Redondance | Traitée par la méthode | Une variable qui double sa voisine a un apport nul quand on la retire : elle sort d'elle-même |

La variable dont l'apport est le plus faible par rapport à son seuil sort, puis on recommence, jusqu'à ce que toutes les variables restantes passent leur seuil.

In [ ]:
SEUIL_GAIN = 0.10                 # une variable doit faire baisser le score d'au moins 0,1 point pour rester
SEUIL_COUT = 0.50                 # une variable coûteuse à obtenir doit apporter au moins 0,5 point
COUTEUSES = {"severite_covid": "téléchargée à la main (OxCGRT), inconnue pour l'avenir"}
seuil = lambda var: SEUIL_COUT if var in COUTEUSES else SEUIL_GAIN

# Élimination pas à pas : à chaque tour, on réentraîne le modèle sans chaque variable restante.
# L'apport d'une variable est la hausse du score quand on la retire. La variable dont l'apport est le plus
# faible par rapport à son seuil sort, si cet apport est sous le seuil ; sinon la sélection s'arrête.
courantes = list(CANDIDATES)
score_courant = valider("Random Forest", courantes, **PARAMS_REF)["score"]
journal = []
while len(courantes) > 1:
    apports = {v: 100 * (valider("Random Forest", [x for x in courantes if x != v], **PARAMS_REF)["score"] - score_courant)
               for v in courantes}
    candidate = min(apports, key=lambda v: apports[v] - seuil(v))
    if apports[candidate] >= seuil(candidate):
        break
    courantes.remove(candidate)
    score_courant += apports[candidate] / 100
    journal.append({"Tour": len(journal) + 1, "Variable retirée": candidate,
                    "Famille": next(f for f, vs in modeles.FAMILLES.items() if candidate in vs),
                    "Apport (points)": apports[candidate],
                    "Raison": COUTEUSES[candidate] + ", apport insuffisant" if candidate in COUTEUSES else "apport inférieur au seuil",
                    "Score après retrait": score_courant})

VARIABLES = courantes
journal = pd.DataFrame(journal)
apport_final = (pd.Series(apports).loc[VARIABLES].sort_values(ascending=False).rename("Apport (points)").to_frame()
                .assign(Famille=lambda d: d.index.map({v: f for f, vs in modeles.FAMILLES.items() for v in vs})))
print("Variables retirées, dans l'ordre :")
display(journal.style.format({"Apport (points)": "{:+.2f}", "Score après retrait": lambda v: pct(v, 2)}))
print("Variables gardées et ce que coûterait le retrait de chacune (dernier tour) :")
display(apport_final.style.format({"Apport (points)": "{:+.2f}"}))
tableaux["5_elimination"] = journal
tableaux["5_variables_retenues"] = apport_final.rename_axis("Variable").reset_index()

apres = valider("Random Forest", VARIABLES, **PARAMS_REF)
print(f"Variables retenues ({len(VARIABLES)} sur {len(CANDIDATES)}) : {', '.join(VARIABLES)}")
print(f"Validation, toutes les variables : {pct(base['court'].mean(), 2)} à un mois, {pct(base['long'].mean(), 2)} à 6 mois, score {pct(base['score'], 2)}")
print(f"Validation, variables retenues  : {pct(apres['court'].mean(), 2)} à un mois, {pct(apres['long'].mean(), 2)} à 6 mois, score {pct(apres['score'], 2)}")

**Lecture.**
- **Cinq variables sortent**, toutes avec un apport négatif : les retirer améliore le modèle. La moyenne sur 3 mois double le mois précédent ; l'âge du catalogue double les lancements sur 12 mois ; le type de boutique et la surface sont couverts par l'effectif et l'historique ; la publicité, répartie presque à parts égales entre les boutiques (rapport L3 + L4), n'apprend rien au modèle.
- **Onze variables restent.** Le mois précédent (+20,6 points) et l'effectif (+3,5) ancrent le niveau. L'indice COVID (+3,4) dépasse largement son seuil de coût : il est gardé, comme marqueur de période. Le taux de promotion (+1,4) porte la saison commerciale.
- **Deux variables d'assortiment restent**, les lancements sur 12 mois (+0,4) et sur 3 mois (+0,2), avec un apport réel mais faible.
- **Le modèle s'améliore** : le score passe de 20,47 % à 19,57 %, surtout à 6 mois (25,25 % → 23,72 %). Moins de variables, mieux choisies, c'est moins de bruit.

## 6. Modèles et réglage des hyperparamètres

| Modèle | Ce qu'il apporte | Réglage |
|---|---|---|
| Références naïves | Le seuil à battre | Aucun |
| Ridge linéaire | Lisible, coefficients interprétables | Grille sur la force de régularisation |
| Ridge quadratique | Courbes et interactions écrites explicitement (variables au carré et produits deux à deux) | Grille |
| Random Forest | Moyenne de nombreux arbres, robuste au bruit | Optuna, 25 essais |
| Gradient boosting (scikit-learn) | Arbres construits l'un après l'autre pour corriger les erreurs | Optuna, 25 essais |
| XGBoost | Bibliothèque de référence du gradient boosting | Optuna, 25 essais |

**Optuna** concentre ses essais là où les scores sont bons, au lieu de tirer au hasard. Chaque essai est noté par le même score de validation que la sélection. Tous les essais sont enregistrés dans **MLflow** (un run par modèle, un sous-run par essai), avec leurs réglages et leurs scores.

In [ ]:
def espace(essai, nom):
    """Hyperparamètres essayés par Optuna pour chaque modèle à arbres."""
    if nom == "Random Forest":
        return dict(n_estimators=essai.suggest_int("n_estimators", 200, 800, step=100),
                    max_depth=essai.suggest_categorical("max_depth", [4, 6, 8, 12, 16, None]),
                    min_samples_leaf=essai.suggest_int("min_samples_leaf", 1, 20),
                    max_features=essai.suggest_categorical("max_features", [1.0, 0.7, 0.5, "sqrt"]))
    if nom == "Gradient boosting (scikit-learn)":
        return dict(learning_rate=essai.suggest_float("learning_rate", 0.01, 0.3, log=True),
                    max_iter=essai.suggest_int("max_iter", 100, 800, step=50),
                    max_leaf_nodes=essai.suggest_int("max_leaf_nodes", 4, 48),
                    min_samples_leaf=essai.suggest_int("min_samples_leaf", 5, 60),
                    l2_regularization=essai.suggest_float("l2_regularization", 1e-4, 10, log=True))
    if nom == "XGBoost":
        return dict(n_estimators=essai.suggest_int("n_estimators", 100, 800, step=50),
                    learning_rate=essai.suggest_float("learning_rate", 0.01, 0.3, log=True),
                    max_depth=essai.suggest_int("max_depth", 2, 8),
                    subsample=essai.suggest_float("subsample", 0.6, 1.0),
                    colsample_bytree=essai.suggest_float("colsample_bytree", 0.5, 1.0),
                    min_child_weight=essai.suggest_float("min_child_weight", 1, 20, log=True),
                    reg_lambda=essai.suggest_float("reg_lambda", 1e-3, 10, log=True))
    raise ValueError(nom)


def noter(r, **params):
    """Paramètres et scores d'un essai dans le run MLflow ouvert."""
    mlflow.log_params({**params, "cible": CIBLE, "nb_variables": len(VARIABLES)})
    mlflow.log_metrics({"wape_un_mois": r["court"].mean(), "wape_6_mois": r["long"].mean(), "score": r["score"]})


resultats_cv, meilleurs_params, runs = {"Naïf (mois précédent, puis dernier mois reconduit)": valider_naif()}, {}, {}

# 1. Ridge : une grille suffit (un seul réglage, la force de la régularisation)
GRILLES = {"Ridge linéaire": [0.01, 0.1, 1, 10, 100, 1000], "Ridge quadratique": [1, 10, 100, 1000, 10000, 100000]}
for nom, alphas in GRILLES.items():
    with mlflow.start_run(run_name=nom) as parent:
        essais = {}
        for alpha in alphas:
            essais[alpha] = valider(nom, VARIABLES, alpha=alpha)
            with mlflow.start_run(run_name=f"{nom} · alpha={alpha}", nested=True):
                noter(essais[alpha], alpha=alpha)
        alpha = min(essais, key=lambda k: essais[k]["score"])
        meilleurs_params[nom], resultats_cv[nom], runs[nom] = {"alpha": alpha}, essais[alpha], parent.info.run_id
        noter(essais[alpha], alpha=alpha, variables=", ".join(VARIABLES))
    print(f"{nom} : meilleur score de validation {pct(essais[alpha]['score'], 2)} (alpha = {alpha})")

# 2. Modèles à arbres : recherche Optuna, chaque essai noté par la validation (moyenne des deux horizons)
for nom in ["Random Forest", "Gradient boosting (scikit-learn)", "XGBoost"]:
    with mlflow.start_run(run_name=nom) as parent:
        def objectif(essai):
            params = espace(essai, nom)
            r = valider(nom, VARIABLES, **params)
            with mlflow.start_run(run_name=f"{nom} · essai {essai.number}", nested=True):
                noter(r, **params)
            return r["score"]
        etude = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=GRAINE))
        etude.optimize(objectif, n_trials=ESSAIS_OPTUNA)
        meilleurs_params[nom] = etude.best_params
        resultats_cv[nom] = valider(nom, VARIABLES, **etude.best_params)
        runs[nom] = parent.info.run_id
        noter(resultats_cv[nom], **etude.best_params, variables=", ".join(VARIABLES))
    print(f"{nom} : meilleur score de validation {pct(resultats_cv[nom]['score'], 2)} après {ESSAIS_OPTUNA} essais")

lignes = []
for nom, r in resultats_cv.items():
    ligne = ligne_scores(nom, r)
    ligne.update({f"Pli court {i + 1}": s for i, s in enumerate(r["court"])})
    ligne.update({f"Pli long {i + 1}": s for i, s in enumerate(r["long"])})
    lignes.append(ligne)
comparaison_cv = pd.DataFrame(lignes).set_index("").sort_values("Score (moyenne)")
display(comparaison_cv.style.format(lambda v: pct(v, 2)))
tableaux["6_comparaison_validation"] = comparaison_cv.reset_index(names="Modèle")
print("Hyperparamètres retenus :")
for nom, p in meilleurs_params.items():
    print(f"  {nom} : {p}")

In [ ]:
appris = comparaison_cv.drop(index=[i for i in comparaison_cv.index if i.startswith("Naïf")])
MEILLEUR = appris["Score (moyenne)"].idxmin()
print(f"Modèle retenu sur la validation : {MEILLEUR} (score {pct(appris['Score (moyenne)'].min(), 2)})")

graphe = comparaison_cv[["À un mois (4 plis)", "À 6 mois (2 plis)"]].rename_axis("Modèle").reset_index().melt(
    id_vars="Modèle", var_name="Horizon", value_name="WAPE")
fig, ax = plt.subplots(figsize=(10, 4.8))
sns.barplot(data=graphe, y="Modèle", x="WAPE", hue="Horizon", order=comparaison_cv.index, palette=[viz.PRINCIPALE, viz.ACCENT], ax=ax)
viz.format_pourcent(ax, "x")
ax.grid(axis="x"); ax.grid(axis="y", visible=False); ax.set_ylabel(""); ax.legend(title="")
viz.titre(ax, "Erreur de validation par modèle et par horizon", f"WAPE moyen des plis · modèles classés par score · retenu : {MEILLEUR}")
viz.enregistrer(fig, "10_comparaison_validation")
plt.show()

**Lecture.**
- **La Random Forest est retenue** (score de 19,34 %), à égalité pratique avec XGBoost (19,42 %). Les deux battent le naïf surtout à 6 mois (23,9 % et 23,4 %, contre 28,2 %).
- **À un mois, aucun modèle ne bat clairement le naïf** (13,88 %). Le mois suivant ressemble beaucoup au mois qui vient de se terminer, et ce qui l'en écarte est en grande partie du bruit.
- **Les Ridge s'effondrent** (43 % et 63 %). Les variables d'assortiment et l'indice COVID suivent une tendance : une droite apprise sur 2019 et 2020 la prolonge au-delà de ce qui a été vu, et l'erreur explose à moyen terme. La version quadratique, avec ses 77 termes, apprend en plus le bruit. Les modèles à arbres, eux, ne prolongent jamais une tendance au-delà de ce qu'ils ont vu.
- **Le réglage retenu est profond** (arbres sans limite de profondeur, feuilles d'une seule ligne, 70 % des variables par découpe) : la moyenne de 300 arbres suffit à lisser.

## 7. Test sur 2021

Le modèle retenu et les autres sont entraînés sur toute la période d'entraînement (février 2019 à décembre 2020), puis jugés **une seule fois** sur 2021. Le test sert à juger, pas à choisir : le choix est déjà fait en partie 6.

### 7.1 À un mois

In [ ]:
def predire(nom, lignes_appr, lignes_prev, variables=None, params=None):
    """Entraîne un modèle sur lignes_appr et prévoit lignes_prev à un mois (le mois précédent réel est connu)."""
    variables = VARIABLES if variables is None else variables
    params = meilleurs_params[nom] if params is None else params
    modele = pipeline(nom, variables, **params).fit(lignes_appr[variables], lignes_appr[colonne_cible(CIBLE)])
    return modele, en_dollars(lignes_prev, modele.predict(lignes_prev[variables]), CIBLE)


def mesures_par_canal(reel, prevision, types):
    """Mesures au global et par canal. Une prévision manquante (« même mois l'an dernier » d'une boutique récente) est écartée."""
    reel, prevision, types = np.asarray(reel, float), np.asarray(prevision, float), np.asarray(types)
    connue = ~np.isnan(prevision)
    lignes = [{"Périmètre": "Toutes les boutiques", **modeles.mesurer(reel[connue], prevision[connue])}]
    for t in ORDRE_TYPES:
        masque = connue & (types == t)
        lignes.append({"Périmètre": t, **modeles.mesurer(reel[masque], prevision[masque])})
    return pd.DataFrame(lignes)


previsions_test = {"Naïf : mois précédent": np.exp(test.log_ca_m1.values),
                   "Naïf : même mois l'an dernier": np.exp(test.log_ca_m12.values)}
modeles_test = {}
for nom in appris.index:
    modeles_test[nom], previsions_test[nom] = predire(nom, entrainement, test)

resultats_test = []
for nom, prev in previsions_test.items():
    m = mesures_par_canal(test.ca_net.values, prev, test.type)
    m.insert(0, "Modèle", nom)
    resultats_test.append(m)
    if nom in runs:                                    # les scores de test rejoignent le run du modèle dans MLflow
        with mlflow.start_run(run_id=runs[nom]):
            for _, r in m.iterrows():
                cle = {"Toutes les boutiques": "total", "En ligne": "en_ligne", "Revendeur": "revendeur", "Magasins": "magasins"}[r["Périmètre"]]
                mlflow.log_metrics({f"test_wape_{cle}": r.WAPE, f"test_mae_{cle}": r.MAE})
resultats_test = pd.concat(resultats_test, ignore_index=True)

global_ = resultats_test[resultats_test["Périmètre"] == "Toutes les boutiques"].set_index("Modèle").drop(columns="Périmètre")
magasins = resultats_test[resultats_test["Périmètre"] == "Magasins"].set_index("Modèle")[["MAE", "WAPE", "R²", "Biais"]]
synthese_test = global_.join(magasins, rsuffix=" (magasins)").sort_values("WAPE")
formats = {c: (lambda v: f"{v:,.0f} $".replace(",", " ")) if c.startswith("MAE") else
           (lambda v: f"{v:.3f}") if c.startswith("R²") else (lambda v: pct(v, 1, signe=True)) if c.startswith("Biais") else (lambda v: pct(v, 1))
           for c in synthese_test.columns}
print(f"Test sur {EXERCICE}, prévision à un mois ({len(test)} lignes boutique × mois) :")
display(synthese_test.style.format(formats))
tableaux["7_test_un_mois"] = resultats_test

In [ ]:
detail = resultats_test[resultats_test.Modèle.isin([MEILLEUR, "Naïf : mois précédent"])].pivot(index="Périmètre", columns="Modèle", values=["MAE", "WAPE"])
detail = detail.reindex(["Toutes les boutiques"] + ORDRE_TYPES)
print(f"{MEILLEUR} et la référence naïve, par canal (test {EXERCICE}, à un mois) :")
display(detail.style.format(lambda v: f"{v:,.0f} $".replace(",", " ") if v > 1 else pct(v, 1)))

**Lecture.**
- **À un mois, la Random Forest ne bat pas le naïf** : 12,2 % d'erreur, contre 11,6 % en reprenant simplement le mois précédent. La validation l'annonçait.
- **Elle fait mieux pour la boutique en ligne** (6,4 % contre 8,8 %), **moins bien pour le revendeur** (10,3 % contre 8,8 %) **et pour les magasins** (19,4 % contre 16,0 %).
- **Les magasins sont prévus trop haut de 16,9 %.** En 2021, ils descendent plus bas que tout ce que le modèle a vu : pendant la période d'entraînement, il n'a jamais connu un semestre sans aucun lancement. Un modèle à arbres ne prévoit pas en dessous des niveaux appris, il les « plafonne » par le bas.
- Le R² dépasse 0,99 au global, mais il est trompeur : il mesure surtout que le modèle sait distinguer la boutique en ligne d'un petit magasin. Le WAPE et le biais disent davantage.

**Pour le suivi mensuel, le naïf « mois précédent » reste la référence.**

### 7.2 À douze mois

C'est l'usage qui compte pour la prévision de l'année et pour le simulateur : à partir de décembre 2020, on prévoit les douze mois de 2021 d'un coup, chaque prévision servant d'historique au mois suivant. Les boutiques ouvertes en cours de 2021 sont exclues, faute de point de départ.

In [ ]:
depart = entrainement[["id_boutique", "date", "log_ca"]]          # historique réel connu fin 2020
futur = test.drop(columns=["log_ca_m1", "log_ca_moy3"])

recursif = {}
for nom in appris.index:
    p = modeles.predire_recursif(modeles_test[nom], VARIABLES, depart, futur, CIBLE)
    recursif[nom] = p.set_index(["id_boutique", "date"]).ca_prevu
base_rec = test.set_index(["id_boutique", "date"])
dernier = depart.sort_values("date").groupby("id_boutique").log_ca.last()
recursif["Naïf : décembre 2020 reconduit"] = np.exp(base_rec.index.get_level_values(0).map(dernier)).to_series(index=base_rec.index)
recursif["Naïf : même mois l'an dernier"] = np.exp(base_rec.log_ca_m12)

communes = base_rec.index
for p in recursif.values():
    communes = communes.intersection(p.dropna().index)
reel_rec = base_rec.loc[communes]
lignes_rec = []
for nom, p in recursif.items():
    m = mesures_par_canal(reel_rec.ca_net.values, p.loc[communes].values, reel_rec.type)
    m.insert(0, "Modèle", nom)
    lignes_rec.append(m)
resultats_recursif = pd.concat(lignes_rec, ignore_index=True)
vue = resultats_recursif.pivot(index="Modèle", columns="Périmètre", values="WAPE")[["Toutes les boutiques"] + ORDRE_TYPES]
biais = resultats_recursif[resultats_recursif["Périmètre"] == "Toutes les boutiques"].set_index("Modèle").Biais
vue["Biais (total)"] = biais
vue = vue.sort_values("Toutes les boutiques")
print(f"Test sur {EXERCICE}, prévision des 12 mois d'un coup à partir de décembre 2020 ({len(communes)} lignes : "
      "boutiques déjà ouvertes fin 2020) · WAPE par canal :")
display(vue.style.format({c: (lambda v: pct(v, 1, signe=True)) if c == "Biais (total)" else (lambda v: pct(v, 1)) for c in vue.columns}))
tableaux["7_test_douze_mois"] = resultats_recursif
with mlflow.start_run(run_id=runs[MEILLEUR]):
    mlflow.log_metric("test_wape_12_mois", vue.loc[MEILLEUR, "Toutes les boutiques"])

In [ ]:
# Le réseau mois par mois : réel 2021 et prévisions à un mois et à douze mois, par canal
suivi = reel_rec.reset_index()[["id_boutique", "date", "type", "ca_net"]].copy()
suivi["À un mois"] = pd.Series(previsions_test[MEILLEUR], index=base_rec.index).loc[communes].values
suivi["À douze mois"] = recursif[MEILLEUR].loc[communes].values
suivi = suivi.rename(columns={"ca_net": "Réel"}).groupby(["type", "date"])[["Réel", "À un mois", "À douze mois"]].sum().reset_index()
long = suivi.melt(id_vars=["type", "date"], var_name="Série", value_name="CA net")

fig, axes = plt.subplots(1, 3, figsize=(15, 4.3), sharex=True)
for ax, t in zip(axes, ORDRE_TYPES):
    sns.lineplot(data=long[long.type == t], x="date", y="CA net", hue="Série", style="Série", markers=True, dashes=False,
                 palette={"Réel": viz.FONCE, "À un mois": viz.ACCENT, "À douze mois": viz.CATEGORIES[2]}, ax=ax, legend=(t == "Magasins"))
    viz.format_dollars(ax); ax.set_xlabel(""); ax.set_ylabel("")
    ax.set_title(t, fontsize=12)
    ax.tick_params(axis="x", rotation=45)
fig.suptitle(f"{MEILLEUR} : chiffre d'affaires net {EXERCICE}, réel et prévu", x=0.01, ha="left", fontweight="bold", color=viz.FONCE)
fig.tight_layout()
viz.enregistrer(fig, "10_test_2021_par_canal")
plt.show()

**Lecture.**
- **À douze mois, la Random Forest est la meilleure** : 12,4 % d'erreur, contre 14,5 % pour « même mois l'an dernier » et 15,7 % pour « décembre 2020 reconduit ». Les deux autres modèles à arbres font moins bien (16,0 % et 19,2 %), et les Ridge divergent (66 % et 121 %), comme en validation.
- **Son avance vient de la boutique en ligne et du revendeur** (6,9 % et 11,3 %). Pour les magasins, elle fait à peu près comme le naïf (18,8 % contre 18,0 %).
- **Le graphique montre où se trouve l'erreur** : en magasin, le modèle prévoit près de 28 M$ par mois de février à juillet, quand le réel tourne autour de 21 à 25 M$. Il ne voit pas venir le creux de février et mars, ni l'érosion qui suit l'arrêt des lancements. Le total 2021 est surestimé de 8 %.

Le modèle est donc utile pour **prévoir une année**, avec une erreur d'environ 12 %, mais il reste optimiste pour les magasins quand l'assortiment se dégrade plus vite que par le passé.

### 7.3 Que valent les variables d'assortiment ?

Vérification de la thèse de l'UA2 par la prévision : le même modèle, avec les mêmes réglages, est entraîné sans les variables d'assortiment, puis comparé sur 2021. C'est une mesure, pas un choix : le test ne sert à rien décider.

In [ ]:
famille_assortiment = [v for v in modeles.FAMILLES["Assortiment"] if v in VARIABLES]
if famille_assortiment:
    sans = [v for v in VARIABLES if v not in famille_assortiment]
    modele_sans, prev_sans = predire(MEILLEUR, entrainement, test, variables=sans)
    rec_sans = modeles.predire_recursif(modele_sans, sans, depart, futur, CIBLE).set_index(["id_boutique", "date"]).ca_prevu
    apport = pd.DataFrame({
        "Avec l'assortiment": [modeles.wape(test.ca_net, previsions_test[MEILLEUR]), modeles.wape(reel_rec.ca_net, recursif[MEILLEUR].loc[communes]),
                               recursif[MEILLEUR].loc[communes].sum() / reel_rec.ca_net.sum() - 1],
        "Sans l'assortiment": [modeles.wape(test.ca_net, prev_sans), modeles.wape(reel_rec.ca_net, rec_sans.loc[communes]),
                               rec_sans.loc[communes].sum() / reel_rec.ca_net.sum() - 1]},
        index=["WAPE à un mois", "WAPE à douze mois", "Biais à douze mois (total prévu / réel − 1)"])
    print(f"{MEILLEUR}, test {EXERCICE} · variables d'assortiment retirées : {', '.join(famille_assortiment)}")
    display(apport.style.format(lambda v: pct(v, 1, signe=True)))
    tableaux["7_apport_assortiment"] = apport
else:
    print("Aucune variable d'assortiment n'a passé la sélection.")

**Lecture.** Sans les lancements, l'erreur monte à un mois (12,2 % → 13,8 %) comme à douze mois (12,4 % → 13,9 %), et la surestimation du total grimpe de +8,0 % à +10,1 %. Les lancements récents aident donc le modèle à suivre la baisse de 2021, ce qui va dans le sens de la thèse de l'UA2. L'écart reste modeste, et il faut le lire avec la réserve de la partie 8 : le modèle ne sait pas séparer l'effet des lancements de celui du temps qui passe.

## 8. Interprétation du modèle retenu

Trois questions : quelles variables le modèle utilise-t-il, dans quel sens jouent-elles, et où se trompe-t-il ?

### 8.1 Importance par permutation sur 2021

In [ ]:
modele_retenu = modeles_test[MEILLEUR]
imp_test = importance_permutation(modele_retenu, test, VARIABLES, repetitions=10).sort_values(ascending=False)
imp_test = imp_test.rename("Hausse du WAPE (points)").rename_axis("Variable").reset_index()
imp_test["Famille"] = imp_test.Variable.map({v: f for f, vs in modeles.FAMILLES.items() for v in vs})
fig, ax = plt.subplots(figsize=(10, 4.5))
sns.barplot(data=imp_test, y="Variable", x="Hausse du WAPE (points)", hue="Famille", dodge=False,
            palette={f: viz.CATEGORIES[i] for i, f in enumerate(modeles.FAMILLES)}, ax=ax)
ax.axvline(0, color=viz.GRIS, lw=1)
ax.grid(axis="x"); ax.grid(axis="y", visible=False); ax.set_ylabel("")
viz.titre(ax, f"Importance des variables sur le test {EXERCICE}", f"{MEILLEUR} · hausse de l'erreur quand on mélange la variable (10 tirages)")
viz.enregistrer(fig, "10_importance_permutation_test")
plt.show()
tableaux["8_importance_test"] = imp_test

**Lecture.** Sur 2021, le modèle s'appuie d'abord sur le **mois précédent** (mélangé, l'erreur gagne 90 points), puis sur l'**effectif** (+17 points), qui situe la taille du magasin, et sur le **taux de promotion** (+3 points), qui porte la saison. Les lancements et l'indice COVID ressortent à zéro, avec le même biais de méthode qu'en partie 5.2 : un mois donné, leur valeur est la même pour toutes les boutiques, et la mélanger ne change presque rien. SHAP, qui ne mélange rien, donne une image plus juste de leur rôle.

### 8.2 SHAP : combien chaque variable pousse chaque prévision

Pour chaque prévision, SHAP répartit l'écart entre la prévision et la moyenne du modèle entre les variables. À gauche, le poids moyen de chaque variable ; à droite, chaque point est une prévision de 2021, placé selon l'effet de la variable (à droite : vers le haut) et coloré selon sa valeur (rouge : élevée).

In [ ]:
preparation = modele_retenu.named_steps["preparation"]
estimateur = modele_retenu.named_steps["modele"]
X_test = pd.DataFrame(preparation.transform(test[VARIABLES]), columns=preparation.get_feature_names_out())
if isinstance(estimateur, Ridge):
    X_appr = pd.DataFrame(preparation.transform(entrainement[VARIABLES]), columns=X_test.columns)
    explicateur = shap.LinearExplainer(estimateur, shap.maskers.Independent(X_appr, max_samples=len(X_appr)))
else:
    explicateur = shap.TreeExplainer(estimateur)
valeurs_shap = explicateur(X_test)

poids = pd.Series(np.abs(valeurs_shap.values).mean(axis=0), index=X_test.columns).sort_values(ascending=False)
poids = (100 * poids).rename("Effet moyen sur la prévision (points de log ≈ %)").rename_axis("Variable").reset_index()
fig, axes = plt.subplots(1, 2, figsize=(15, 5), gridspec_kw={"width_ratios": [1, 1.25]})
sns.barplot(data=poids, y="Variable", x=poids.columns[1], color=viz.PRINCIPALE, ax=axes[0])
axes[0].grid(axis="x"); axes[0].grid(axis="y", visible=False); axes[0].set_ylabel("")
viz.titre(axes[0], "Poids moyen de chaque variable", "Moyenne des |valeurs SHAP|, test 2021")
plt.sca(axes[1])
shap.plots.beeswarm(valeurs_shap, max_display=len(X_test.columns), show=False, plot_size=None, color_bar_label="Valeur de la variable")
axes[1].set_title("Sens de l'effet, ligne par ligne", loc="left", fontweight="bold", color=viz.FONCE)
axes[1].set_xlabel("Valeur SHAP : effet sur le log du CA prévu" if CIBLE == "niveau" else "Valeur SHAP : effet sur la variation prévue (log)")
fig.tight_layout()
viz.enregistrer(fig, "10_shap")
plt.show()
tableaux["8_shap"] = poids

**Lecture.**
- **Le mois précédent** pèse le plus (environ 20 points de log en moyenne) : il porte la taille de la boutique. Les points rouges très à droite sont la boutique en ligne et le revendeur.
- **L'indice COVID vient ensuite** (13 points). En 2021, il est élevé partout et tire toutes les prévisions vers le bas : le modèle l'utilise pour dire « nous sommes après la rupture de 2020 ».
- **Le taux de promotion** (6 points) : les mois sans promotion (bleu, mai à juillet) poussent vers le haut, ceux à 20 % (rouge, octobre à janvier) vers le bas. Ce n'est pas un effet de la promotion, que le notebook 07 n'a pas trouvé : c'est la saison, que le calendrier promotionnel suit à l'identique chaque année.
- **Les lancements sur 12 mois** (4 points) jouent un rôle réel, plus faible que celui de l'indice COVID avec lequel ils se confondent.

In [ ]:
# Une prévision expliquée : le magasin et le mois où le modèle se trompe le plus en 2021 (en dollars)
ecarts = test.assign(prevision=previsions_test[MEILLEUR], ecart=lambda d: d.prevision - d.ca_net)
pire = ecarts[ecarts.type == "Magasins"].ecart.abs().idxmax()
ligne = ecarts.loc[pire]
contributions = pd.Series(valeurs_shap.values[pire], index=X_test.columns).sort_values(key=np.abs, ascending=False)
print(f"{ligne.nom_boutique}, {ligne.date:%Y-%m} : réel {m_dollars(ligne.ca_net, 2)}, prévu {m_dollars(ligne.prevision, 2)} "
      f"(mois précédent : {m_dollars(np.exp(ligne.log_ca_m1), 2)})")
print(f"Écart au mois précédent : prévu {pct(ligne.prevision / np.exp(ligne.log_ca_m1) - 1, 1, signe=True)}, "
      f"réel {pct(ligne.ca_net / np.exp(ligne.log_ca_m1) - 1, 1, signe=True)}")
print("Contributions SHAP (en points de log : +10 ≈ +10 % sur la prévision), les six plus fortes :")
display((100 * contributions).head(6).rename("Contribution (points de log)").to_frame())

**Lecture d'un cas.** Le magasin de Rome, en mars 2021, est l'erreur la plus forte en dollars : 0,54 M$ prévus pour 0,25 M$ réalisés. Le modèle prévoyait une hausse de 69 % par rapport à février, le magasin a reculé de 20 %. SHAP montre pourquoi : l'indice COVID tire vers le bas (−9 points), mais la saison portée par le taux de promotion pousse vers le haut (+8 points), et le modèle a appris sur 2019 et 2020 que mars repart après février. En 2021, le creux de février et mars, expliqué dans l'UA2, est plus profond et plus long : le modèle n'en a vu aucun exemple de cette ampleur.

### 8.3 Forme de la relation apprise pour l'assortiment

On fixe une variable d'assortiment à chaque valeur d'une grille, toutes les autres variables restant réelles, et on regarde comment bouge la prévision moyenne. C'est une **vérification de cohérence** du modèle, pas un scénario : les scénarios chiffrés sont réservés aux notebooks 13 et 14.

In [ ]:
def dependance(modele, lignes, variable, variables, points=25):
    """Variation prévue moyenne quand on fixe `variable` à chaque valeur d'une grille (les autres variables restent réelles)."""
    grille = np.linspace(lignes[variable].quantile(0.02), lignes[variable].quantile(0.98), points)
    sortie = []
    for valeur in grille:
        x = lignes[variables].copy()
        x[variable] = valeur
        sortie.append(modele.predict(x).mean())
    sortie = np.array(sortie)
    centre = sortie[len(sortie) // 2]                       # valeur médiane de la variable = référence
    return pd.DataFrame({"valeur": grille, "écart": np.exp(sortie - centre) - 1})


a_tracer = [v for v in modeles.FAMILLES["Assortiment"] if v in VARIABLES] or [imp_test.Variable.iloc[0]]
fig, axes = plt.subplots(1, len(a_tracer), figsize=(5.2 * len(a_tracer), 4), squeeze=False)
libelles = {"lancements_3m": "Produits lancés sur les 3 derniers mois", "lancements_12m": "Produits lancés sur les 12 derniers mois",
            "age_catalogue": "Âge moyen du catalogue (mois)"}
for ax, var in zip(axes[0], a_tracer):
    d = dependance(modele_retenu, entrainement, var, VARIABLES)
    sns.lineplot(data=d, x="valeur", y="écart", color=viz.PRINCIPALE, lw=2.5, ax=ax)
    sns.rugplot(x=entrainement[var].drop_duplicates(), color=viz.GRIS, height=0.04, ax=ax)
    viz.format_pourcent(ax, "y", 1)
    ax.axhline(0, color=viz.GRIS, lw=1)
    ax.set_xlabel(libelles.get(var, var)); ax.set_ylabel("Prévision moyenne, écart à la valeur médiane")
fig.suptitle("Forme de la relation apprise pour l'assortiment", x=0.01, ha="left", fontweight="bold", color=viz.FONCE)
fig.tight_layout()
viz.enregistrer(fig, "10_forme_assortiment")
plt.show()

**Lecture : un résultat important pour la suite.**
- **La relation apprise va à l'envers de la thèse.** Plus il y a de lancements sur 12 mois, plus la prévision baisse : environ +4 % autour de 450 lancements, −7 % autour de 620. Pour les lancements sur 3 mois, la courbe monte et descend sans logique commerciale.
- **Ce n'est pas un effet, c'est une confusion.** Les lancements, l'indice COVID et l'âge du catalogue évoluent ensemble de 2019 à 2021 (partie 2). Avec 23 mois d'entraînement, le modèle ne peut pas savoir laquelle de ces variables fait bouger les ventes. Il répartit l'effet au mieux pour prévoir, pas pour expliquer.
- **Conséquence pour le simulateur (notebook 14)** : ce modèle prévoit correctement une année au fil de l'eau, mais il **ne doit pas servir à chiffrer le levier « rythme de lancement »**. Faire varier les lancements dans ce modèle donnerait un effet de sens contraire à ce qu'ont montré les notebooks 06b et 06. Le chiffrage de ce levier doit partir du mécanisme mesuré dans l'UA2 (le chiffre d'affaires apporté par les nouveautés, 69 % de la chute des magasins), pas de la courbe d'un modèle prédictif.

C'est la différence entre **prévoir** et **expliquer** : un bon modèle de prévision n'est pas un modèle causal.

### 8.4 Où le modèle se trompe-t-il ?

In [ ]:
ecarts["mois_nom"] = ecarts.date.dt.month.map(lambda m: NOMS_MOIS[m - 1])
par_mois = ecarts.groupby(["type", "mois_nom"], sort=False).apply(lambda g: modeles.wape(g.ca_net, g.prevision), include_groups=False).rename("WAPE").reset_index()
par_boutique = (ecarts[ecarts.type == "Magasins"].groupby("nom_boutique")
                .apply(lambda g: pd.Series({"WAPE": modeles.wape(g.ca_net, g.prevision), "Biais": g.prevision.sum() / g.ca_net.sum() - 1,
                                            "CA net 2021": g.ca_net.sum()}), include_groups=False)
                .sort_values("WAPE", ascending=False))

fig, axes = plt.subplots(1, 2, figsize=(15, 4.6), gridspec_kw={"width_ratios": [1.3, 1]})
sns.barplot(data=par_mois, x="mois_nom", y="WAPE", hue="type", hue_order=ORDRE_TYPES, palette=COULEURS_TYPES, ax=axes[0])
viz.format_pourcent(axes[0]); axes[0].set_xlabel(""); axes[0].legend(title="")
viz.titre(axes[0], "Erreur par mois et par canal", f"WAPE à un mois, test {EXERCICE}")
sns.histplot(par_boutique.WAPE, bins=15, color=viz.PRINCIPALE, ax=axes[1])
axes[1].axvline(modeles.wape(ecarts[ecarts.type == "Magasins"].ca_net, ecarts[ecarts.type == "Magasins"].prevision), color=viz.ACCENT, lw=2)
viz.format_pourcent(axes[1], "x"); axes[1].set_ylabel("Magasins"); axes[1].set_xlabel("WAPE du magasin sur 2021")
viz.titre(axes[1], "Erreur magasin par magasin", "Trait orange : WAPE de l'ensemble des magasins")
fig.tight_layout()
viz.enregistrer(fig, "10_erreurs")
plt.show()

print("Les cinq magasins les moins bien prévus :")
display(par_boutique.head(5).style.format({"WAPE": lambda v: pct(v), "Biais": lambda v: pct(v, 1, signe=True), "CA net 2021": m_dollars}))
tableaux["8_erreurs_par_mois"] = par_mois
tableaux["8_erreurs_par_magasin"] = par_boutique.reset_index()

**Lecture.**
- **Par mois** : en magasin, l'erreur est la plus forte de janvier à avril (22 % à 32 %), puis redescend autour de 12 % à 20 %. C'est la période du creux de 2021. Le revendeur suit le même profil (34 % en janvier, sous 10 % à partir de mai). La boutique en ligne reste sous 16 % tous les mois.
- **Par magasin** : la plupart des magasins sont prévus avec 15 % à 25 % d'erreur. Une poignée se détache entre 35 % et 61 %, toujours **par excès** (Venise, Marseille, Berlin, Rome, Düsseldorf). Ce sont des magasins dont les ventes de 2021 tombent bien plus bas que leur historique : le modèle, qui ne prévoit pas en dessous de ce qu'il a vu, les plafonne.
- **À examiner avec le métier** : ces cinq magasins ont-ils perdu plus de références que les autres en 2021 ? Le notebook 06b montrait une baisse uniforme du réseau, avec deux exceptions : il faudra vérifier s'il s'agit des mêmes.

## 9. Modèle final et prévision 2022 « au fil de l'eau »

Une fois le modèle jugé, il est réentraîné avec les mêmes réglages sur **tout** l'historique disponible (février 2019 à décembre 2021) : le test a servi à juger la méthode, et 2021 apporte l'information la plus récente.

**Hypothèses au fil de l'eau**, c'est-à-dire si rien ne change :

| Variable | Hypothèse 2022 |
|---|---|
| Calendrier | Le vrai calendrier 2022 (jours, fins de semaine) |
| Jours fériés, indice COVID | Les valeurs du même mois de 2021, pays par pays |
| Promotion, publicité | Le calendrier et les budgets de 2021, boutique par boutique |
| Lancements | Le plan de 2021 reconduit : 272 produits, aux mêmes dates, donc rien au second semestre |
| Historique | Prévu mois après mois à partir du réel de décembre 2021 |

C'est le **scénario de référence** auquel le simulateur comparera les leviers.

In [ ]:
# 1. Modèle final : mêmes réglages, entraîné sur les 35 mois disponibles (février 2019 à décembre 2021)
modele_final = pipeline(MEILLEUR, VARIABLES, **meilleurs_params[MEILLEUR]).fit(table[VARIABLES], table[colonne_cible(CIBLE)])

# 2. Les 12 mois de 2022, pour chaque boutique ouverte en décembre 2021
ouvertes = table[table.date == table.date.max()]
mois_2022 = pd.date_range(f"{EXERCICE + 1}-01-01", periods=12, freq="MS")
futur_2022 = ouvertes[["id_boutique", "nom_boutique", "type_boutique", "type", "pays", "surface", "effectif"]].merge(
    pd.DataFrame({"date": mois_2022}), how="cross")
futur_2022["annee"], futur_2022["mois"] = futur_2022.date.dt.year, futur_2022.date.dt.month
futur_2022["mois_sin"] = np.sin(2 * np.pi * futur_2022.mois / 12)
futur_2022["mois_cos"] = np.cos(2 * np.pi * futur_2022.mois / 12)
futur_2022["jours"] = futur_2022.date.dt.days_in_month
futur_2022["jours_week_end"] = futur_2022.date.map(modeles._jours_week_end)

# Hypothèses « au fil de l'eau » : tout ce qui n'est pas calendaire reprend le même mois de 2021.
# Une boutique ouverte en cours de 2021 n'a pas tous ses mois : on prend alors la valeur de son pays
# (jours fériés, COVID), le taux le plus fréquent du réseau (promotion) ou la médiane de son type (publicité).
dernier = table[table.annee == EXERCICE]
cles_bm = ["id_boutique", "mois"]
futur_2022 = futur_2022.merge(dernier[cles_bm + ["taux_promotion", "publicite"]], on=cles_bm, how="left")
futur_2022 = futur_2022.merge(dernier.groupby(["pays", "mois"])[["jours_feries", "severite_covid"]].first().reset_index(),
                              on=["pays", "mois"], how="left")
reseau_promo = dernier.groupby("mois").taux_promotion.agg(lambda s: s.mode().iloc[0])
type_pub = dernier.groupby(["type_boutique", "mois"]).publicite.median()
futur_2022["taux_promotion"] = futur_2022.taux_promotion.fillna(futur_2022.mois.map(reseau_promo))
futur_2022 = futur_2022.merge(type_pub.rename("publicite_type").reset_index(), on=["type_boutique", "mois"], how="left")
futur_2022["publicite"] = futur_2022.publicite.fillna(futur_2022.publicite_type)

# Lancements : le plan de 2021 reconduit à l'identique, aux mêmes dates un an plus tard
lancements = dim_produit.date_mise_en_vente.dropna()
lancements_2021 = lancements[lancements.dt.year == EXERCICE]
plan_2022 = pd.concat([lancements, lancements_2021 + pd.DateOffset(years=1)], ignore_index=True)
futur_2022 = futur_2022.merge(modeles.variables_catalogue(plan_2022, mois_2022), left_on="date", right_index=True)
print("Valeurs manquantes dans les variables de 2022 :", int(futur_2022[[v for v in CANDIDATES if v not in HISTORIQUE]].isna().sum().sum()))

historique = table[["id_boutique", "date", "log_ca"]]
prevision_2022 = modeles.predire_recursif(modele_final, VARIABLES, historique, futur_2022, CIBLE)
print(f"Prévision {EXERCICE + 1} pour {prevision_2022.id_boutique.nunique()} boutiques · lancements reconduits : {len(lancements_2021)}")

annuel = pd.concat([table.groupby(["annee", "type"]).ca_net.sum(),
                    prevision_2022.groupby(["annee", "type"]).ca_prevu.sum()]).unstack("type")[ORDRE_TYPES]
annuel["Total"] = annuel.sum(axis=1)
evolution = annuel.pct_change().iloc[-1]
display(annuel.map(m_dollars))
print(f"Évolution {EXERCICE} → {EXERCICE + 1} prévue au fil de l'eau : " +
      " · ".join(f"{c} {pct(evolution[c], 1, signe=True)}" for c in annuel.columns))
print("Attention : le CA 2019 compte 11 mois (janvier n'a pas de mois précédent dans la table de modélisation).")
tableaux["9_annuel_2022"] = annuel.reset_index()

In [ ]:
reel_mensuel = commun.assign(type=commun.type_boutique.map(LIBELLES_TYPES)).groupby(["type", "date"]).ca_net.sum().rename("CA net").reset_index().assign(Série="Réel")
prev_mensuel = prevision_2022.groupby(["type", "date"]).ca_prevu.sum().rename("CA net").reset_index().assign(Série="Prévision au fil de l'eau")
serie = pd.concat([reel_mensuel, prev_mensuel], ignore_index=True)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.3), sharex=True)
for ax, t in zip(axes, ORDRE_TYPES):
    sns.lineplot(data=serie[serie.type == t], x="date", y="CA net", hue="Série", style="Série",
                 palette={"Réel": viz.FONCE, "Prévision au fil de l'eau": viz.ACCENT}, ax=ax, legend=(t == "Magasins"))
    ax.axvspan(mois_2022[0], mois_2022[-1], color=viz.CLAIR, zorder=0)
    viz.format_dollars(ax); ax.set_xlabel(""); ax.set_ylabel(""); ax.set_title(t, fontsize=12)
    ax.set_ylim(bottom=0)
    ax.xaxis.set_major_locator(mdates.YearLocator()); ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
fig.suptitle(f"Chiffre d'affaires net mensuel 2019 à {EXERCICE}, et {EXERCICE + 1} si rien ne change", x=0.01, ha="left", fontweight="bold", color=viz.FONCE)
fig.tight_layout()
viz.enregistrer(fig, "10_prevision_2022")
plt.show()

**Lecture.**
- **Au fil de l'eau, 2022 ressemble à 2021** : 684,7 M$, soit −0,5 %, avec la même baisse légère dans les trois canaux. Le modèle prolonge le niveau atteint fin 2021 et la saison habituelle.
- **Cette prévision est probablement optimiste pour les magasins.** Sur 2021, le même modèle surestimait les magasins (+17 % à un mois) parce qu'il ne prévoit pas de baisse plus forte que celles qu'il a vues. Si l'assortiment continue de vieillir au rythme de 2021, la baisse des magasins devrait se poursuivre plutôt que se stabiliser.
- **Lecture pour la direction** : même sans nouvelle dégradation visible dans le modèle, l'entreprise ne retrouve pas en 2022 le niveau de 2019 (890,7 M$ sur onze mois). Le statu quo fige la situation de 2021, dans laquelle les magasins sont déficitaires.

## 10. Enregistrement

| Sortie | Contenu | Utilisée par |
|---|---|---|
| `data/outputs/predictions/previsions_ventes.parquet` | Réel, prévision et erreur par boutique et par mois : test 2021 à un mois et à douze mois, prévision 2022 | Application Streamlit, rapport |
| `data/outputs/predictions/modele_prevision.joblib` | Modèle final, liste des variables, cible, réglages, version de scikit-learn | Simulateur 14, application Streamlit |
| `mlruns/` | Tous les essais (réglages, scores de validation et de test), le modèle final et les figures | Consultation avec `mlflow ui` |
| `reports/livrables/L5_rapport_final/prevision_tableaux.xlsx` | Tous les tableaux du notebook | Rapport final |

Pour consulter les essais, depuis le dossier du projet : `mlflow ui --backend-store-uri sqlite:///mlruns/mlflow.db`, puis ouvrir http://127.0.0.1:5000. Le message de MLflow sur le format « cloudpickle » rappelle seulement qu'un modèle enregistré ne doit être rouvert que s'il vient d'une source de confiance, ce qui est le cas ici.

In [ ]:
# 1. Prévisions : test 2021 (à un mois et à douze mois) et 2022 au fil de l'eau
cles = ["id_boutique", "nom_boutique", "type", "date", "annee", "mois"]
test_un = test[cles + ["ca_net"]].assign(prevision=previsions_test[MEILLEUR], horizon="1 mois", jeu=f"Test {EXERCICE}")
test_douze = (test[cles + ["ca_net"]].set_index(["id_boutique", "date"]).join(recursif[MEILLEUR].rename("prevision"), how="inner")
              .reset_index().assign(horizon="12 mois", jeu=f"Test {EXERCICE}"))
futur_sortie = prevision_2022[cles].assign(ca_net=np.nan, prevision=prevision_2022.ca_prevu.values, horizon="12 mois",
                                           jeu=f"Prévision {EXERCICE + 1} au fil de l'eau")
sortie = pd.concat([test_un, test_douze, futur_sortie], ignore_index=True)
sortie["erreur"] = sortie.prevision - sortie.ca_net
sortie = sortie.rename(columns={"ca_net": "ca_net_reel"})
chemin_prev = config.PREDICTIONS / "previsions_ventes.parquet"
sortie.to_parquet(chemin_prev, index=False)

# 2. Le modèle, avec tout ce qu'il faut pour le réutiliser (simulateur 14, application Streamlit)
paquet = {"modele": modele_final, "nom": MEILLEUR, "variables": VARIABLES, "cible": CIBLE,
          "parametres": meilleurs_params[MEILLEUR], "entraine_jusqu_a": f"{table.date.max():%Y-%m}",
          "version_scikit_learn": sklearn.__version__,
          "lecture": ("prévision = exp(modele.predict(X))" if CIBLE == "niveau" else "prévision = exp(log_ca_m1 + modele.predict(X))")
                     + " ; sur plusieurs mois : src/modeles.predire_recursif"}
chemin_modele = config.PREDICTIONS / "modele_prevision.joblib"
joblib.dump(paquet, chemin_modele)

# 3. Le modèle final dans MLflow, avec son exemple d'entrée
with mlflow.start_run(run_name=f"Modèle final : {MEILLEUR}"):
    mlflow.log_params({**meilleurs_params[MEILLEUR], "variables": ", ".join(VARIABLES), "cible": CIBLE,
                       "entraine_jusqu_a": paquet["entraine_jusqu_a"]})
    mlflow.log_metrics({"test_wape_1_mois": modeles.wape(test.ca_net, previsions_test[MEILLEUR]),
                        "test_wape_12_mois": vue.loc[MEILLEUR, "Toutes les boutiques"]})
    mlflow.sklearn.log_model(modele_final, name="modele", input_example=table[VARIABLES].head(5),
                             serialization_format="cloudpickle")
    for figure in sorted(config.FIGURES.glob("10_*.png")):
        mlflow.log_artifact(str(figure), artifact_path="figures")

# 4. Les tableaux du notebook, pour le rapport
chemin_excel = analyse.exporter_tableaux(tableaux, config.L5 / "prevision_tableaux.xlsx")
for chemin in [chemin_prev, chemin_modele, chemin_excel]:
    print("Enregistré :", chemin.relative_to(config.ROOT))
print("Runs MLflow de l'expérience :", len(mlflow.search_runs(experiment_names=[EXPERIENCE])))

## 11. Synthèse

### 1. Le modèle retenu

| Élément | Choix |
|---|---|
| Modèle | Random Forest (300 arbres, 70 % des variables par découpe) |
| Cible | Log du chiffre d'affaires net du mois, par boutique |
| Variables | 11 sur 16 : mois précédent, effectif, indice COVID (marqueur de période), taux de promotion, lancements sur 12 et sur 3 mois, cinq variables de calendrier |
| Choisi par | Validation glissante sur 2019 et 2020, moyenne de l'erreur à un mois et à 6 mois |

### 2. Ce qu'il vaut sur 2021

| Horizon | Random Forest | Meilleure référence naïve | Verdict |
|---|---|---|---|
| Un mois | 12,2 % | 11,6 % (mois précédent) | Le naïf suffit pour le suivi mensuel |
| Douze mois | 12,4 % | 14,5 % (même mois l'an dernier) | Le modèle est utile pour prévoir l'année |
| Douze mois, magasins | 18,8 % | 18,0 % (décembre reconduit) | Pas mieux que le naïf ; surestimation des magasins |

### 3. Ce qu'il faut retenir
1. **Prévoir le mois suivant est difficile** dans ces données : la boutique d'un mois ressemble à celle du mois précédent, et l'écart est en grande partie du bruit. Aucun des sept modèles ne bat nettement la référence naïve à un mois.
2. **À l'échelle de l'année, le modèle apporte** : il suit mieux la saison et le niveau de chaque canal que les références naïves, surtout pour la boutique en ligne et le revendeur.
3. **Il reste optimiste pour les magasins** quand leurs ventes tombent plus bas que tout ce qu'il a vu, comme en 2021. Sa prévision 2022 au fil de l'eau (684,7 M$, −0,5 %) est à lire comme un plafond pour les magasins.
4. **Il prévoit, il n'explique pas.** Les lancements, l'indice COVID et l'âge du catalogue évoluent ensemble : le modèle ne peut pas isoler l'effet des nouveautés, et la relation qu'il a apprise va même à l'envers. Il ne doit pas servir à chiffrer le levier « rythme de lancement ».
5. **Méthode** : la sélection des variables doit se faire en réentraînant le modèle sans chaque variable. La permutation sous-estime fortement les variables qui ne changent qu'avec le temps (indice COVID : 0 point par permutation, 3,4 points par élimination).

### 4. Pour la suite
- **Notebook 14 (simulateur)** : utiliser ce modèle pour la trajectoire de référence au fil de l'eau, et chiffrer le levier des lancements à partir du mécanisme mesuré dans l'UA2 (chiffre d'affaires des nouveautés), pas en faisant varier les lancements dans ce modèle.
- **Notebook 12 (typologie des magasins)** : regarder si les magasins les plus surestimés (Venise, Marseille, Berlin, Rome, Düsseldorf) forment un groupe à part.
- **Avec plus d'historique** (au moins trois années complètes avant la période à prévoir), l'effet des lancements pourrait être estimé directement ; c'est une limite des données, pas de la méthode.